## Serialize a Raster Data using SchemaOrg and the CUAHSI.org-Raster extension

The purpose of this notebook is to evaluate how Raster data can be extracted and mapped to our Pydantic classes.

In [1]:
import sys

# add the parent directory to the path. This is the 
# directory that contains our pydantic classes.
sys.path.append('..')

# import schema classes
import base
import core
import dataset
import raster


import rasterio
from pyproj import CRS

In [2]:

def encode_raster_metadata(filepath, validate_bbox=True):
    # Open the Raster file
    src = rasterio.open(filepath)
    
    # Cell Information
    cell_columns = src.width
    cell_rows = src.height
    cell_data_type = src.dtypes[0]
    cell_x_size = src.transform.a 
    cell_y_size = abs(src.transform.e)
    
    # Bands
    num_bands = src.count
    band_data = {}
    for band in range(1, num_bands+1):
        dat = src.read(band)
        band_meta = dict(
            min_value = dat.min().item(),
            max_value = dat.max().item(),
            nan_value = src.nodatavals[band - 1]
        )
        band_data[band] = band_meta
    
    # get crs metadata
    crs = CRS.from_wkt(src.crs.wkt)
    crs_name = crs.name
    crs_datum = crs.datum.name
    crs_unit = crs.axis_info[0].unit_name
    crs_string = crs.to_wkt()
    
    # Extent
    extent_west, extent_south, extent_east, extent_north = src.bounds
    
    # Other metadata
    name = src.name.split('/')[1:][0]
    
    src.close()


    # Encode metadata
    box_str = f'{extent_south} {extent_west} {extent_north} {extent_east}'
    geo = base.GeoShape(box = box_str, validate_bbox=validate_bbox)

    crs_type = "Geographic Coordinate System" if crs.is_geographic else "Projected Coordinate System"
    srs = [
        base.PropertyValue(
        name = crs_type,
        value = crs_name),
        base.PropertyValue(
        name = "Datum",
        value = crs_datum),
        base.PropertyValue(
        name = "Unit",
        value = crs_unit),
        base.PropertyValue(
        name = "Coordinate String",
        value = crs_string),
    ]
    
    
    place = base.Place(
        geo=geo,
        additionalProperty=srs,
    )

    gridvariables = []
    for band_idx in band_data.keys():
        variable = raster.GridVariable(
            name = f'Band {band_idx}',
            minValue = band_data[band_idx]['min_value'],
            maxValue = band_data[band_idx]['max_value'],
            noDataValue = band_data[band_idx]['nan_value'],
            band = band_idx)
        gridvariables.append(variable)
    
    r = raster.GeographicRaster(
        rows = cell_rows,
        columns = cell_columns,
        xCellSize = cell_x_size,
        yCellSize = cell_y_size,
        cellValueType = cell_data_type,
        variableMeasured = gridvariables,
        spatialCoverage=place,
    )

    return r

Test on single band GeoTiff

In [3]:
meta = encode_raster_metadata('data/Onion3ad8o.tif', validate_bbox=True)
print(meta.model_dump_json(exclude_none=True, indent=4))

{
    "context": "https://schema.org/",
    "type": "GeographicRaster",
    "spatialCoverage": {
        "type": "Place",
        "geo": {
            "type": "GeoShape",
            "box": "30.01462962962525 -98.30768518519098 30.27027777889825 -97.57953703383897"
        },
        "additionalProperty": [
            {
                "type": "PropertyValue",
                "name": "Geographic Coordinate System",
                "value": "NAD83"
            },
            {
                "type": "PropertyValue",
                "name": "Datum",
                "value": "North American Datum 1983"
            },
            {
                "type": "PropertyValue",
                "name": "Unit",
                "value": "Degree"
            },
            {
                "type": "PropertyValue",
                "name": "Coordinate String",
                "value": "GEOGCRS[\"NAD83\",DATUM[\"North American Datum 1983\",ELLIPSOID[\"GRS 1980\",6378137,298.257222101,LENGTHUNIT[\"me

Test on a multi-band GeoTiff

In [4]:
src = rasterio.open('data/landsat-multiband-sample.tif')
print(f'Number of Bands: {src.count}')
src.close()

Number of Bands: 6


In [5]:
meta = encode_raster_metadata('data/landsat-multiband-sample.tif', validate_bbox=False)
print(meta.model_dump_json(exclude_none=True, indent=4))

{
    "context": "https://schema.org/",
    "type": "GeographicRaster",
    "spatialCoverage": {
        "type": "Place",
        "geo": {
            "type": "GeoShape",
            "box": "4504185.0 254685.0 4743315.0 490215.0"
        },
        "additionalProperty": [
            {
                "type": "PropertyValue",
                "name": "Projected Coordinate System",
                "value": "WGS 84 / UTM zone 15N"
            },
            {
                "type": "PropertyValue",
                "name": "Datum",
                "value": "World Geodetic System 1984"
            },
            {
                "type": "PropertyValue",
                "name": "Unit",
                "value": "metre"
            },
            {
                "type": "PropertyValue",
                "name": "Coordinate String",
                "value": "PROJCRS[\"WGS 84 / UTM zone 15N\",BASEGEOGCRS[\"WGS 84\",DATUM[\"World Geodetic System 1984\",ELLIPSOID[\"WGS 84\",6378137,298.25722356

Test on ASCII Raster

In [6]:
meta = encode_raster_metadata('data/Onion3ad8o.tif')
print(meta.model_dump_json(exclude_none=True, indent=4))

{
    "context": "https://schema.org/",
    "type": "GeographicRaster",
    "spatialCoverage": {
        "type": "Place",
        "geo": {
            "type": "GeoShape",
            "box": "30.01462962962525 -98.30768518519098 30.27027777889825 -97.57953703383897"
        },
        "additionalProperty": [
            {
                "type": "PropertyValue",
                "name": "Geographic Coordinate System",
                "value": "NAD83"
            },
            {
                "type": "PropertyValue",
                "name": "Datum",
                "value": "North American Datum 1983"
            },
            {
                "type": "PropertyValue",
                "name": "Unit",
                "value": "Degree"
            },
            {
                "type": "PropertyValue",
                "name": "Coordinate String",
                "value": "GEOGCRS[\"NAD83\",DATUM[\"North American Datum 1983\",ELLIPSOID[\"GRS 1980\",6378137,298.257222101,LENGTHUNIT[\"me